<a href="https://colab.research.google.com/github/asiaihsan/MovieAkinator/blob/main/MovieAkinator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!wget https://datasets.imdbws.com/title.basics.tsv.gz
!wget https://datasets.imdbws.com/title.ratings.tsv.gz

--2026-11-08 19:20:46--  https://datasets.imdbws.com/title.basics.tsv.gz
Resolving datasets.imdbws.com (datasets.imdbws.com)... 13.226.53.33, 13.226.53.109, 13.226.53.113, ...
Connecting to datasets.imdbws.com (datasets.imdbws.com)|13.226.53.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 230046569 (219M) [binary/octet-stream]
Saving to: ‘title.basics.tsv.gz’

title.basics.tsv.gz 100%[===================>] 219.39M   166MB/s    in 1.3s    

2026-11-08 19:20:48 (166 MB/s) - ‘title.basics.tsv.gz’ saved [230046569/230046569]

--2026-11-08 19:20:48--  https://datasets.imdbws.com/title.ratings.tsv.gz
Resolving datasets.imdbws.com (datasets.imdbws.com)... 13.226.53.33, 13.226.53.109, 13.226.53.113, ...
Connecting to datasets.imdbws.com (datasets.imdbws.com)|13.226.53.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8741674 (8.3M) [binary/octet-stream]
Saving to: ‘title.ratings.tsv.gz’

title.ratings.tsv.g 100%[===================>]   8.

In [4]:
import pandas as pd

basics = pd.read_csv("title.basics.tsv.gz", sep="\t", na_values="\\N", low_memory=False)
ratings = pd.read_csv("title.ratings.tsv.gz", sep="\t", na_values="\\N")

print(basics.shape)
print(ratings.shape)
basics.head()

(12947333, 9)
(1730098, 3)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [5]:
movies = basics[basics["titleType"] == "movie"]
movies = movies.merge(ratings, on="tconst")
movies = movies[movies["numVotes"] >= 50000]

print(movies.shape)
movies[["primaryTitle", "startYear", "genres", "numVotes"]].head()

(4688, 11)


,primaryTitle,startYear,genres,numVotes
1387,The Cabinet of Dr. Caligari,1920.0,"Horror,Mystery,Thriller",77486
1856,The Kid,1921.0,"Comedy,Drama,Family",147770
2110,Nosferatu,1922.0,"Fantasy,Horror",125249
2618,Sherlock Jr.,1924.0,"Action,Comedy,Romance",67530
2706,Battleship Potemkin,1925.0,"Drama,History,Thriller",65958


In [6]:
movies = movies.dropna(subset=["startYear", "genres"]).copy()
movies["startYear"] = movies["startYear"].astype(int)
movies["decade"] = (movies["startYear"] // 10) * 10

genre_cols = movies["genres"].str.get_dummies(sep=",")

print(movies.shape)
print(genre_cols.columns.tolist())

(4687, 12)
['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']


In [7]:
features = genre_cols.copy()
for d in sorted(movies["decade"].unique()):
    features[f"decade_{d}"] = (movies["decade"] == d).astype(int)

def best_question(candidates):
    share = features.loc[candidates].mean()
    return (share - 0.5).abs().idxmin()

print(best_question(movies.index))

Drama


In [8]:
for t in [6.5, 7.5, 8.0, 8.5]:
    features[f"rating_{t}+"] = (movies["averageRating"] >= t).astype(int)

for m in [100, 120, 140, 160]:
    features[f"runtime_{m}+"] = (movies["runtimeMinutes"].astype(float) >= m).astype(int)

for y in [1990, 2000, 2005, 2010, 2015]:
    features[f"year_{y}+"] = (movies["startYear"] >= y).astype(int)

print(features.shape)

(4687, 47)


In [ ]:
!wget -q https://datasets.imdbws.com/title.principals.tsv.gz
!wget -q https://datasets.imdbws.com/name.basics.tsv.gz

In [ ]:
keep = set(movies["tconst"])
parts = []
for chunk in pd.read_csv("title.principals.tsv.gz", sep="\t", na_values="\\N",
                         usecols=["tconst", "ordering", "nconst", "category"],
                         chunksize=1_000_000):
    mask = chunk["tconst"].isin(keep) & (
        (chunk["category"] == "director") |
        (chunk["category"].isin(["actor", "actress"]) & (chunk["ordering"] <= 4))
    )
    parts.append(chunk[mask])
people = pd.concat(parts)

counts = people["nconst"].value_counts()
popular = counts[counts >= 6].index
names = pd.read_csv("name.basics.tsv.gz", sep="\t", na_values="\\N",
                    usecols=["nconst", "primaryName"])
names = names[names["nconst"].isin(popular)].set_index("nconst")["primaryName"]

print(len(popular))
print(names.head(10).tolist())

In [ ]:
p = people[people["nconst"].isin(popular)][["tconst", "nconst"]].drop_duplicates()
pm = pd.crosstab(p["tconst"], p["nconst"]).clip(upper=1)
pm = pm.reindex(movies["tconst"].values, fill_value=0)
pm.index = movies.index
pm.columns = ["person_" + names[c] for c in pm.columns]
pm = pm.loc[:, ~pm.columns.duplicated()]

features = pd.concat([features, pm], axis=1)
print(features.shape)


In [26]:
features = genre_cols.copy()

for y in [1980, 1990, 2000, 2010, 2020]:
    features[f"year_{y}"] = (movies["startYear"] >= y).astype(int)

hours = movies["runtimeMinutes"].astype(float) / 60
for h in [1.5, 2, 2.5]:
    features[f"hours_{h}"] = (hours >= h).astype(int)

for pct, name in [(0.8, "famous"), (0.95, "very_famous")]:
    cutoff = movies["numVotes"].quantile(pct)
    features[name] = (movies["numVotes"] >= cutoff).astype(int)

def ask_text(col):
    if col.startswith("person_"):
        return f"Is {col[7:]} in it, or did they direct it?"
    if col.startswith("year_"):
        return f"Was it released in {col[5:]} or later?"
    if col.startswith("hours_"):
        return f"Is it longer than {col[6:]} hours?"
    if col == "famous":
        return "Is it a very well-known movie?"
    if col == "very_famous":
        return "Is it one of the most famous movies ever?"
    return f"Is it a {col} movie?"

def play():
    mistakes = pd.Series(0.0, index=features.index)
    asked = []
    people_asked = 0
    scale = {"y": 1.0, "p": 0.75, "pn": 0.25, "n": 0.0}
    while len(asked) < 25:
        alive = mistakes[mistakes <= mistakes.min() + 2].index
        if len(alive) <= 1:
            break
        share = features.loc[alive].mean().drop(asked, errors="ignore")
        is_person = share.index.str.startswith("person_")
        ps = share[is_person]
        ps = ps[ps > 0]
        rest = share[~is_person]
        use_people = len(alive) <= 150 and len(asked) % 2 == 0 and people_asked < 6
        pool = ps if (use_people and len(ps) > 0) else rest
        q = (pool - 0.5).abs().idxmin()
        if q.startswith("person_"):
            people_asked += 1
        asked.append(q)
        answer = ""
        while answer not in ("y", "p", "s", "pn", "n"):
            answer = input(f"{ask_text(q)} (y / p = probably / s = not sure / pn = probably not / n): ").strip().lower()
        if answer != "s":
            mistakes += (features[q] - scale[answer]).abs()
    ranked = mistakes.to_frame("m").join(movies["numVotes"])
    best = ranked.sort_values(["m", "numVotes"], ascending=[True, False]).index[:3]
    print(movies.loc[best, "primaryTitle"].tolist())


In [ ]:
play()

In [18]:
print(features.shape)
print([c for c in features.columns if c.startswith("person_")][:5])


(4687, 33)
[]


In [22]:
!wget -q -nc https://datasets.imdbws.com/title.principals.tsv.gz
!wget -q -nc https://datasets.imdbws.com/name.basics.tsv.gz

keep = set(movies["tconst"])
parts = []
for chunk in pd.read_csv("title.principals.tsv.gz", sep="\t", na_values="\\N",
                         usecols=["tconst", "ordering", "nconst", "category"],
                         chunksize=1_000_000):
    mask = chunk["tconst"].isin(keep) & (
        (chunk["category"] == "director") |
        (chunk["category"].isin(["actor", "actress"]) & (chunk["ordering"] <= 4))
    )
    parts.append(chunk[mask])
people = pd.concat(parts)

counts = people["nconst"].value_counts()
popular = counts[counts >= 6].index
names = pd.read_csv("name.basics.tsv.gz", sep="\t", na_values="\\N",
                    usecols=["nconst", "primaryName"])
names = names[names["nconst"].isin(popular)].set_index("nconst")["primaryName"]

print(len(popular))

929


In [25]:
p = people[people["nconst"].isin(popular)][["tconst", "nconst"]].drop_duplicates()
pm = pd.crosstab(p["tconst"], p["nconst"]).clip(upper=1)
pm = pm.reindex(movies["tconst"].values, fill_value=0)
pm.index = movies.index
pm.columns = ["person_" + names[c] for c in pm.columns]
pm = pm.loc[:, ~pm.columns.duplicated()]

features = pd.concat([features, pm], axis=1)
print(features.shape)

(4687, 1891)
